# Data Science Now: Hands-On Workshop (Solutions)
**DLSU Lipa · WCTP 2026 · Jude Teves**

In Notebook 1 you saw data science, 2017-style: a CSV, features, a random forest, a slide. Today you'll build the 2026 version: an assistant that answers questions about **Philippine laws** from real Wikipedia articles, and you'll take it through the five stages of a modern AI workflow.

| Lab | Stage | What you'll do |
|---|---|---|
| 1 | **Documents → data** | Load articles and explore them like any dataset |
| 2 | **LLMs as data tools** | Turn messy text into a clean table, then *grade* the LLM |
| 3 | **RAG** | Embeddings, cosine similarity, and a ChromaDB vector database |
| 4 | **Evaluation** | Measure whether it actually works (spoiler: not always) |
| 5 | **Agents** | Let the LLM use tools to solve multi-step questions |
| 🏁 | **Challenge** | Group work: make it better, then try to break it |

**How this notebook works**
1. `File → Save a copy in Drive` so your changes are saved.
2. Run cells top to bottom with `Shift + Enter`. No GPU needed.
3. ✏️ **TODO** sections are marked `# ✏️ YOUR CODE HERE`. Replace the `raise NotImplementedError(...)` line with your code (usually 1–3 lines). Hints sit right above.
4. This is the **solutions** copy: every ✏️ TODO is already filled in.
5. 🧪 **TRY IT** cells are for experimenting.

> **The big idea:** your data science fundamentals (pandas, vectors, evaluation, skepticism) don't go away with AI. They're what turn an LLM demo into a system that works.

## 0 · Setup
Run both cells. The first takes 1–2 minutes (it installs ChromaDB and a small embedding model).

**About the LLM:** inside Colab we use Colab's built-in AI library (`google.colab.ai`): free, and **no API key**. Outside Colab, the cell falls back to a free Gemini API key: create one at [aistudio.google.com](https://aistudio.google.com/apikey) and add it in Colab's 🔑 **Secrets** panel as `GEMINI_API_KEY`.

In [ ]:
#@title ⚙️ Install & import libraries
!pip install -q chromadb sentence-transformers google-genai

import json, re, time, warnings, ast, operator, io
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 100)

DATA_URLS = ["https://raw.githubusercontent.com/Cyntwikip/LLM/main/ds-rag/data/", "https://cdn.jsdelivr.net/gh/Cyntwikip/LLM@main/ds-rag/data/", "https://talks.judeteves.com/dlsu-lipa-ds-ai/data/"]   # GitHub, then two backups

def fetch_data(filename):
    # Download a workshop data file, trying each mirror in turn.
    for base in DATA_URLS:
        try:
            r = requests.get(base + filename, timeout=30)
            r.raise_for_status()
            return r.content
        except Exception as e:
            print(f"  (mirror failed: {base[:40]}… {type(e).__name__})")
    raise RuntimeError(f"Could not download {filename}. Check your internet connection.")

# Friendly message when a ✏️ TODO hasn't been done yet
try:
    def _todo_hint(shell, etype, evalue, tb, tb_offset=None):
        print(f"✏️  Not done yet: {evalue}")
        print("    Find the '# ✏️ YOUR CODE HERE' section, replace the raise line with your code,"
              " or check the solutions notebook.")
    get_ipython().set_custom_exc((NotImplementedError,), _todo_hint)
except NameError:
    pass

print("Ready ✅")

In [ ]:
#@title 🔌 Connect to an LLM
MODEL = None                       # None = Colab's default. Run ai.list_models() to see options.
GEMINI_MODEL = "gemini-flash-latest"   # only used by the API-key fallback

BACKEND = None
try:
    from google.colab import ai    # Colab's built-in AI: free, no key needed
    BACKEND = "colab"
    try:
        print("Models you can use:", ai.list_models())
    except Exception:
        pass
except Exception:
    pass

if BACKEND is None:
    import os
    key = os.environ.get("GEMINI_API_KEY")
    try:
        from google.colab import userdata
        key = key or userdata.get("GEMINI_API_KEY")
    except Exception:
        pass
    if key:
        from google import genai
        client = genai.Client(api_key=key)
        BACKEND = "gemini"

def llm(prompt: str, retries: int = 4) -> str:
    # Send a prompt, get text back. Waits and retries if the service is busy.
    if BACKEND is None:
        raise RuntimeError("No LLM connected. Run this in Colab, or add GEMINI_API_KEY to Secrets.")
    for attempt in range(retries):
        try:
            if BACKEND == "colab":
                out = ai.generate_text(prompt, model_name=MODEL) if MODEL else ai.generate_text(prompt)
            else:
                out = client.models.generate_content(model=MODEL or GEMINI_MODEL, contents=prompt).text
            return (out or "").strip()
        except Exception as e:
            wait = 5 * (attempt + 1)
            print(f"  ⏳ LLM busy ({type(e).__name__}); retrying in {wait}s…")
            time.sleep(wait)
    raise RuntimeError("LLM call kept failing. Wait a minute and re-run the cell.")

print(f"Backend: {BACKEND}")
print(llm("In one short, upbeat sentence, welcome data science students from DLSU Lipa to an AI workshop."))

---
## Lab 1 · Documents → Data
Most real-world data isn't a neat CSV. It's reports, policies, contracts, web pages. Our dataset is a snapshot of **12 Wikipedia articles about Philippine laws** that affect you directly: data privacy, cybercrime, free tuition, SIM registration, anti-hazing and more.

*(Snapshot taken November 2026 so everyone sees the same text. Licensed [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/).)*

In [ ]:
snapshot = json.loads(fetch_data("ph_laws_wikipedia.json"))
DROP_SECTIONS = ("See also", "References", "External links", "Notes", "Further reading", "Sources")

def clean(text):
    # Cut off reference sections, then tidy whitespace.
    for sec in DROP_SECTIONS:
        text = re.split(rf"\n=+ {re.escape(sec)} =+\n", text)[0]
    return re.sub(r"\n{3,}", "\n\n", text).strip()

docs = pd.DataFrame(snapshot["articles"])[["title", "url", "text"]]
docs["text"] = docs["text"].apply(clean)
print(f"Loaded {len(docs)} articles · retrieved {snapshot['retrieved'][:10]}")
docs.head(3)

### ✅ TODO 1: Treat text like any other dataset
Before touching AI, do what a data scientist always does: **look at the data.** Add a column `n_words` with the number of words in each article.

*Hint:* pandas string methods: `.str.split()` turns each text into a list of words, and `.str.len()` counts list items.

In [ ]:
# ✏️ YOUR CODE HERE (TODO 1): add a column `n_words` with the number of words in each article
docs["n_words"] = docs["text"].str.split().str.len()
# ✏️ END OF YOUR CODE

ax = docs.sort_values("n_words").plot.barh(x="title", y="n_words", legend=False, figsize=(8, 5), color="#2D5F4F")
ax.set_xlabel("Words"); ax.set_ylabel(""); ax.set_title("How much text do we have per article?")
plt.tight_layout(); plt.show()

### ✅ TODO 2: Classic text mining still works
Which Republic Act numbers does each article mention? A regular expression finds them with no AI at all. The function is written for you. **Apply it** to every article's text to make a new column `ra_mentions`.

*Hint:* `docs["text"].apply(some_function)`

In [ ]:
def find_ra_numbers(text):
    return sorted(set(re.findall(r"(?:Republic Act|R\.A\.|RA)\s*(?:No\.?\s*)?(\d{4,5})", text)))

# ✏️ YOUR CODE HERE (TODO 2): add a column `ra_mentions` by running find_ra_numbers on every article
docs["ra_mentions"] = docs["text"].apply(find_ra_numbers)
# ✏️ END OF YOUR CODE
docs[["title", "n_words", "ra_mentions"]]

---
## Lab 2 · LLMs as Data Tools
Regex works when patterns are rigid. But *"who signed this law, and when?"* is phrased differently in every article. This is where LLMs shine: **turning unstructured text into structured data.**

Read the prompt below. It's a **spec**, not a chat: exact fields, exact format, and an instruction **not to guess**.

In [ ]:
EXTRACT_PROMPT = '''You are a careful data extractor. Read the article and extract facts about the
main Philippine law it describes. Return ONLY a JSON object with these keys:
  "law_name": official short name of the law
  "ra_number": Republic Act number, digits only (e.g. "10173")
  "date_signed": date signed into law, as YYYY-MM-DD
  "signed_by": name of the president who signed it
  "topic": one of ["privacy & tech", "education", "health", "safety & rights", "economy", "governance", "other"]
  "one_line_summary": one plain-English sentence a student would understand
Use null for anything the article does not state. Do not guess.

ARTICLE TITLE: {title}
ARTICLE:
{text}
'''

def parse_json(s):
    # LLMs sometimes wrap JSON in ```fences``` or add chatter. Grab the first {...} block.
    m = re.search(r"\{.*\}", s, flags=re.S)
    return json.loads(m.group(0)) if m else {}

### ✅ TODO 3: Call the LLM
Complete `extract_law`: fill the prompt template with the article's `title` and (shortened) `text`, send it to `llm(...)`, and parse the reply with `parse_json(...)`.

*Hint:* `EXTRACT_PROMPT.format(title=..., text=...)`

In [ ]:
def extract_law(row, max_words=1500):
    text = " ".join(row.text.split()[:max_words])     # keep prompts short (and cheap)
    # ✏️ YOUR CODE HERE (TODO 3): set `out` to the LLM's reply, parsed into a dictionary
    out = parse_json(llm(EXTRACT_PROMPT.format(title=row.title, text=text)))
    # ✏️ END OF YOUR CODE
    out["source"] = row.title
    return out

# Always test on ONE example before running on everything.
extract_law(docs.iloc[1])

In [ ]:
records = []
for _, row in docs.iterrows():
    try:
        records.append(extract_law(row)); print("✓", row.title)
    except Exception as e:
        print("✗", row.title, e)

laws = pd.DataFrame(records)
laws["date_signed"] = pd.to_datetime(laws.get("date_signed"), errors="coerce")
laws

You just turned 12 messy web pages into a table, and from here it's ordinary pandas:

In [ ]:
print(laws["signed_by"].value_counts(dropna=False), "\n")
timeline = laws.dropna(subset=["date_signed"]).sort_values("date_signed")
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.scatter(timeline["date_signed"], range(len(timeline)), color="#C4542A")
for i, (_, r) in enumerate(timeline.iterrows()):
    ax.annotate(str(r["law_name"]), (r["date_signed"], i), xytext=(6, -3), textcoords="offset points", fontsize=8)
ax.set_yticks([]); ax.set_title("When were these laws signed?")
plt.tight_layout(); plt.show()

### ⚠️ Can you trust the table? Grade the LLM like any model
An LLM extracting data is still a *model making predictions*. In 2017 you'd never ship a classifier without a test set. Same here. We hand-checked every law against its article and saved a **reference table**. Let's score the LLM field by field.

In [ ]:
reference = pd.read_csv(io.BytesIO(fetch_data("laws_reference.csv")), dtype=str)
reference

### ✅ TODO 4: Field-level accuracy
`compare` lines up the LLM's answer and the reference for one field. Finish the `match` column: `True` when the two values are equal after normalizing.

*Hint:* use the `norm` helper on both sides: `norm(r.llm) == norm(r.ref)`

In [ ]:
def norm(x):
    if x is None or (isinstance(x, float) and np.isnan(x)) or str(x).lower() in ("nat", "nan", "none", ""):
        return ""
    return re.sub(r"[^a-z0-9]", "", str(x).lower())

def compare(field):
    llm_vals = laws.set_index("source")[field]
    if field == "date_signed":
        llm_vals = llm_vals.dt.strftime("%Y-%m-%d")
    df = reference.set_index("source")[[field]].rename(columns={field: "ref"}).join(llm_vals.rename("llm"))
    # ✏️ YOUR CODE HERE (TODO 4): add a True/False column `df["match"]`: does the normalized LLM value equal the normalized reference?
    df["match"] = df.apply(lambda r: norm(r.llm) == norm(r.ref), axis=1)
    # ✏️ END OF YOUR CODE
    df["note"] = reference.set_index("source")["note"].fillna("")
    return df

for field in ["ra_number", "date_signed"]:
    print(f"{field:12s} accuracy: {compare(field)['match'].mean():.0%}")
compare("date_signed")

🧪 **TRY IT:** look at the rows where `match` is `False`, and read the `note` column. Is the LLM wrong, or did it "know" something the article never says? (Sometimes `null` is the *correct* answer: one law here was never signed at all.) This habit of **checking model output against ground truth** is the foundation of Lab 4.

---
## Lab 3 · Retrieval-Augmented Generation (RAG)
### First, the problem
Ask the LLM about our laws **without** the documents (*closed-book*):

In [ ]:
closed_book_questions = [
    "What is the Republic Act number of the E-Governance Act of the Philippines?",
    "How many senators voted in favor of the Safe Spaces Act on third reading?",
]
for q in closed_book_questions:
    print("Q:", q)
    print("A:", llm(f"Answer in one or two sentences: {q}"), "\n")

It might be right, confidently wrong, or out of date. **You can't tell from the answer alone.** RAG fixes this: **retrieve** relevant passages from *your* documents, then ask the model to answer **only from them, with citations**.

```
question ──► embed ──► vector DB (ChromaDB) ──► top-k chunks ──► prompt ──► LLM ──► answer + [citations]
```

### Step 1: Chunking
Retrieval works better on small pieces. Split each article into overlapping chunks of words.

### ✅ TODO 5: Make the chunk
Inside the loop, `i` is the starting word. Build `body`: the words from position `i` up to `i + chunk_words`, joined back into one string with spaces.

*Hint:* list slicing `words[a:b]` and `" ".join(...)`

In [ ]:
CHUNK_WORDS = 150     # 🧪 knob: try 80, 150, 300
OVERLAP     = 30      # 🧪 knob: words shared between neighbouring chunks
ADD_TITLE   = True    # 🧪 knob: prefix each chunk with its article title?

def make_chunks(docs, chunk_words=CHUNK_WORDS, overlap=OVERLAP, add_title=ADD_TITLE):
    out = []
    for _, d in docs.iterrows():
        words = d.text.split()
        step = max(1, chunk_words - overlap)
        for i in range(0, max(1, len(words) - overlap), step):
            # ✏️ YOUR CODE HERE (TODO 5): set `body` to the words from position i up to i + chunk_words, joined into one string
            body = " ".join(words[i:i + chunk_words])
            # ✏️ END OF YOUR CODE
            out.append({"title": d.title, "chunk_id": len(out),
                        "text": f"[{d.title}] {body}" if add_title else body})
    return pd.DataFrame(out)

chunks = make_chunks(docs)
print(f"{len(docs)} documents → {len(chunks)} chunks")
chunks.sample(3, random_state=1)

### Step 2: Embeddings, under the hood
An **embedding model** turns text into a vector (here, 384 numbers) so that passages with similar *meaning* end up close together, even when they share no words. We measure "close" with **cosine similarity**, the same dot product you know from linear algebra.

In [ ]:
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer("all-MiniLM-L6-v2")    # small, fast, runs on CPU

E = embedder.encode(chunks.text.tolist(), normalize_embeddings=True, show_progress_bar=True)
print("Embedding matrix:", E.shape, " ← one row per chunk")

### ✅ TODO 6: Cosine similarity by hand
The vectors are normalized (length 1), so cosine similarity is just a **dot product**. Compute `scores`: the similarity between every chunk vector in `E` and the query vector `q`.

*Hint:* the matrix–vector product `E @ q` gives one score per chunk.

In [ ]:
query = "Can I get in trouble for catcalling someone on the street?"    # 🧪 try your own!
q = embedder.encode([query], normalize_embeddings=True)[0]

# ✏️ YOUR CODE HERE (TODO 6): set `scores` to the cosine similarity between every row of E and the query vector q
scores = E @ q
# ✏️ END OF YOUR CODE
top = np.argsort(scores)[::-1][:3]
for i in top:
    print(f"{scores[i]:.3f}  {chunks.title[i]:40s} {chunks.text[i][len(chunks.title[i])+3:][:80]}…")

Notice the query never says *"Safe Spaces Act"* or *"harassment"*, yet the top hits are probably about it. That's meaning, not keywords.

### Step 3: A real vector database (ChromaDB)
Looping over a NumPy matrix works for 100 chunks. Real systems hold millions, so they use a **vector database** that stores vectors and searches them fast. ChromaDB is a popular open-source one, and it runs right here in memory.

### ✅ TODO 7: Add the chunks to ChromaDB
Call `col.add(...)` with four lists: `ids` and `metadatas` (already prepared), `documents` (the chunk texts) and `embeddings` (the vectors in `E`).

*Hint:* `chunks.text.tolist()` and `E.tolist()`

In [ ]:
import chromadb
chroma = chromadb.Client()                       # in-memory database

def build_collection(chunks, E, name="ph_laws"):
    try:
        chroma.delete_collection(name)           # start fresh when re-running
    except Exception:
        pass
    col = chroma.create_collection(name, embedding_function=None, configuration={"hnsw": {"space": "cosine"}})
    # ✏️ YOUR CODE HERE (TODO 7): add the chunks to `col` with col.add(...). `ids` and `metadatas` are ready for you
    ids = chunks.chunk_id.astype(str).tolist()
    metadatas = [{"title": t} for t in chunks.title]
    col.add(ids=ids, documents=chunks.text.tolist(), embeddings=E.tolist(), metadatas=metadatas)
    # ✏️ END OF YOUR CODE
    return col

collection = build_collection(chunks, E)
print("Chunks in ChromaDB:", collection.count())

### ✅ TODO 8: Query the database
Embed the question, then ask ChromaDB for the `k` nearest chunks.

*Hint:* `collection.query(query_embeddings=[...], n_results=k)`. Chroma wants a *list* of query vectors, even for one query.

In [ ]:
class ChromaRetriever:
    name = "ChromaDB"
    def __init__(self, collection):
        self.col = collection
    def search(self, question, k=4):
        q_emb = embedder.encode([question], normalize_embeddings=True)[0].tolist()
        # ✏️ YOUR CODE HERE (TODO 8): set `res` to ChromaDB's k nearest chunks for the query vector q_emb
        res = self.col.query(query_embeddings=[q_emb], n_results=k)
        # ✏️ END OF YOUR CODE
        return pd.DataFrame({"title": [m["title"] for m in res["metadatas"][0]],
                             "text": res["documents"][0],
                             "score": [1 - d for d in res["distances"][0]]})   # distance → similarity

chroma_ret = ChromaRetriever(collection)
chroma_ret.search("Can I get in trouble for catcalling someone on the street?", k=3)

### Baseline: TF-IDF keyword search
For a fair fight in Lab 4, here's the classic keyword retriever from text mining class. (Already written. Just run it.)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

class TfidfRetriever:
    name = "TF-IDF"
    def __init__(self, chunks):
        self.chunks = chunks
        self.vec = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), sublinear_tf=True)
        self.M = self.vec.fit_transform(chunks.text)
    def search(self, question, k=4):
        s = (self.M @ self.vec.transform([question]).T).toarray().ravel()
        top = s.argsort()[::-1][:k]
        return self.chunks.iloc[top][["title", "text"]].assign(score=s[top]).reset_index(drop=True)

tfidf = TfidfRetriever(chunks)
tfidf.search("Can I get in trouble for catcalling someone on the street?", k=3)

### Step 4: Generation, grounded and cited

### ✅ TODO 9: Send the RAG prompt
`sources` is already built from the retrieved chunks. Fill in the prompt template and call the LLM.

*Hint:* `llm(RAG_PROMPT.format(sources=..., question=...))`

In [ ]:
RAG_PROMPT = '''Answer the question using ONLY the numbered sources below.
- Cite the sources you used like [1] or [2][3].
- If the sources do not contain the answer, reply exactly: "I don't know based on the documents."
- Keep it to 1-3 sentences.

SOURCES:
{sources}

QUESTION: {question}
ANSWER:'''

def rag_answer(question, retriever=None, k=4, verbose=False):
    retriever = retriever or chroma_ret
    hits = retriever.search(question, k=k)
    sources = "\n\n".join(f"[{i+1}] {t}" for i, t in enumerate(hits.text))
    # ✏️ YOUR CODE HERE (TODO 9): set `answer` to the LLM's reply to the filled-in RAG prompt
    answer = llm(RAG_PROMPT.format(sources=sources, question=question))
    # ✏️ END OF YOUR CODE
    if verbose:
        print("Q:", question, "\nA:", answer)
        for i, t in enumerate(hits.title):
            print(f"   [{i+1}] {t}")
    return {"answer": answer, "sources": list(hits.title)}

for q in closed_book_questions:
    rag_answer(q, verbose=True); print()

🧪 **TRY IT:** ask your own questions, including one the documents *can't* answer (e.g. *"What's the penalty for jaywalking?"*). Does it admit it doesn't know?

In [ ]:
rag_answer("Is a 'like' or a retweet of a libelous post a crime under the Cybercrime law?", verbose=True);

---
## Lab 4 · Evaluation: does it actually work?
A few good-looking answers prove nothing. Below is a test set: questions with known answers (each checked against the articles), plus two *trap* questions the documents can't answer.

In [ ]:
EVAL = pd.DataFrame([
    # question, acceptable answers (any match = correct), source article, type
    ("What is the Republic Act number of the SIM Registration Act?", ["11934"], "SIM Registration Act", "fact"),
    ("What is the Safe Spaces Act colloquially known as?", ["bastos"], "Safe Spaces Act", "fact"),
    ("Which government body was created under the Data Privacy Act of 2012?", ["national privacy commission"], "National Privacy Commission", "fact"),
    ("On what date did President Aquino approve the Cybercrime Prevention Act of 2012?", ["september 12 2012", "12 september 2012"], "Cybercrime Prevention Act of 2012", "fact"),
    ("For how many days did the Supreme Court's October 2012 restraining order stop the Cybercrime Prevention Act?", ["120"], "Cybercrime Prevention Act of 2012", "fact"),
    ("Which justice penned the 2014 Supreme Court ruling in Disini v. Secretary of Justice?", ["abad"], "Disini v. Secretary of Justice", "fact"),
    ("What was the Supreme Court's vote in Disini v. Secretary of Justice?", ["12-1-2"], "Disini v. Secretary of Justice", "fact"),
    ("What does Republic Act 10931 provide for students in state universities and colleges?", ["free tuition", "tuition-free", "free higher education"], "Universal Access to Quality Tertiary Education Act", "fact"),
    ("When did President Duterte sign the Universal Health Care Act?", ["february 20 2019", "20 february 2019"], "Universal Health Care Act (Philippines)", "fact"),
    ("Which earlier Republic Act did the Anti-Hazing Act of 2018 amend?", ["8049"], "Anti-Hazing Act of 2018", "fact"),
    ("What is the Republic Act number of the E-Governance Act?", ["12254"], "E-Governance Act", "fact"),
    ("Besides Philippine citizens, who must register under the Philippine Identification System Act?", ["resident alien"], "Philippine Identification System Act", "fact"),
    ("How many senators voted in favor of the Safe Spaces Act on third reading?", ["21"], "Safe Spaces Act", "fact"),
    ("What is the maximum fine under the Philippine Clean Air Act of 1999?", [], None, "trap"),
    ("Who is the current chairperson of the Commission on Elections?", [], None, "trap"),
], columns=["question", "accept", "source", "type"])

REFUSALS = ["don't know", "do not know", "not in the", "not contain", "no information", "not mentioned", "cannot find", "can't find", "not provided"]

def normalize(s):
    s = str(s).lower().replace("–", "-").replace("—", "-")
    return re.sub(r"[,\.]", "", s)

print(f"{len(EVAL)} test questions ({(EVAL.type == 'trap').sum()} traps)")

### ✅ TODO 10: The grader
For **trap** questions, the answer is correct if it admits it doesn't know (that part is written). For **fact** questions, finish the line: the answer is correct if **any** acceptable answer appears in it (after normalizing both).

*Hint:* `any(normalize(x) in a for x in accept)`

In [ ]:
def grade(answer, accept, qtype):
    a = normalize(answer)
    if qtype == "trap":
        return any(r in a for r in REFUSALS)
    # ✏️ YOUR CODE HERE (TODO 10): return True if any acceptable answer (normalized) appears in `a`
    return any(normalize(x) in a for x in accept)
    # ✏️ END OF YOUR CODE

# Quick sanity checks: your grader should print True, False, True
print(grade("It is Republic Act No. 11934 [1].", ["11934"], "fact"),
      grade("It is RA 9999.", ["11934"], "fact"),
      grade("I don't know based on the documents.", [], "trap"))

### 4a · Evaluate the retriever alone (no LLM calls)
If the right passage never reaches the LLM, the LLM can't answer. **Hit rate@k** = how often the correct article appears in the top *k* results. Think of it as *recall*, for search.

In [ ]:
def hit_rate(retriever, k):
    facts = EVAL[EVAL.type == "fact"]
    return np.mean([row.source in set(retriever.search(row.question, k=k).title) for _, row in facts.iterrows()])

ks = [1, 2, 3, 4, 5, 8]
hr = pd.DataFrame({r.name: [hit_rate(r, k) for k in ks] for r in (tfidf, chroma_ret)}, index=ks)
hr.index.name = "k"
display(hr.style.format("{:.0%}"))
hr.plot(marker="o", ylim=(0, 1.05), title="Retrieval hit rate@k", figsize=(6, 3.5), color=["#9C9A94", "#2D5F4F"]); plt.show()

🤔 **Surprised that old-school TF-IDF does so well?** Look at the test questions: almost every one *names the law* ("…the SIM Registration Act…"), and exact keyword matching loves that. Embeddings shine when people **don't** use the official words, like the catcalling question. Real users rarely know the official name of a law.

That's a classic evaluation lesson: **your score is only as good as your test set.** (Mission B is your chance to write questions that are more realistic. Many production systems combine both methods in *hybrid search*.)

### 4b · Evaluate the whole system
Three systems, same questions: **closed-book**, **RAG + TF-IDF**, **RAG + ChromaDB**. *(About 45 LLM calls, 1–3 minutes.)*

In [ ]:
def closed_book(q):
    return llm(f"Answer in 1-2 sentences. If you are not sure, say \"I don't know\".\nQuestion: {q}")

SYSTEMS = {
    "Closed-book":     closed_book,
    "RAG + TF-IDF":    lambda q: rag_answer(q, retriever=tfidf)["answer"],
    "RAG + ChromaDB":  lambda q: rag_answer(q, retriever=chroma_ret)["answer"],
}

def run_eval(systems=SYSTEMS, eval_df=EVAL):
    rows = []
    for name, fn in systems.items():
        print(f"Running {name}…")
        for _, row in eval_df.iterrows():
            ans = fn(row.question)
            rows.append({"system": name, "question": row.question, "type": row.type,
                         "answer": ans, "correct": grade(ans, row.accept, row.type)})
    return pd.DataFrame(rows)

results = run_eval()
scores = results.pivot_table(index="system", columns="type", values="correct", aggfunc="mean")
scores["overall"] = results.groupby("system")["correct"].mean()
display(scores.style.format("{:.0%}"))

In [ ]:
scores["overall"].reindex(SYSTEMS.keys()).plot.bar(color=["#9C9A94", "#8FC1AE", "#2D5F4F"], ylim=(0, 1),
                                                   title="Accuracy on our test set", rot=0, figsize=(6, 3.5))
plt.show()

# Read the failures. This is where you learn the most.
results[~results.correct][["system", "question", "answer"]]

### 4c · Who grades the grader?
Keyword grading is transparent but strict (*"twelve thousand two hundred fifty-four"* would fail). A common alternative is **LLM-as-a-judge**, but that's another model that can be wrong. Compare them:

In [ ]:
JUDGE_PROMPT = '''You are grading an answer to a quiz question.
Question: {q}
Expected answer (any of): {accept}
Student answer: {a}
If the expected answer list is empty, the correct behaviour is to say the information is unavailable.
Reply with exactly one word: CORRECT or INCORRECT.'''

sub = results[results.system == "RAG + ChromaDB"].merge(EVAL[["question", "accept"]], on="question")
sub["judge"] = ["INCORRECT" not in llm(JUDGE_PROMPT.format(q=r.question, accept=r.accept, a=r.answer)).upper()
                for _, r in sub.iterrows()]
print(f"Keyword grader and LLM judge agree on {(sub.correct == sub.judge).mean():.0%} of answers")
sub[sub.correct != sub.judge][["question", "answer", "correct", "judge"]]

> **Takeaway:** accuracy, precision and recall still matter. For AI *systems* you also evaluate retrieval, refusals, grounding, and even your evaluator. **Evaluation is a data science job.**

---
## Lab 5 · From answering to *doing*: Agents
*(No TODOs here. Run and read.)*

Some questions need **several steps and different tools**:
> *"How many years apart were the Data Privacy Act and the Safe Spaces Act signed?"*

That needs the table (dates), then arithmetic. An **agent** is an LLM in a loop: it **decides** which tool to use, **observes** the result, and repeats until it can answer.
```
question ──► LLM thinks ──► picks a tool ──► tool runs ──► observation ──┐
                ▲                                                        │
                └────────────────────────────────────────────────────────┘   … until final answer
```

In [ ]:
# ── Tools are just Python functions ──────────────────────────────────────────
def search_documents(query: str) -> str:
    return "\n\n".join(chroma_ret.search(query, k=3).text)

def laws_table(_: str = "") -> str:
    cols = ["law_name", "ra_number", "date_signed", "signed_by"]
    t = laws[cols].copy()
    t["date_signed"] = pd.to_datetime(t["date_signed"], errors="coerce").dt.strftime("%Y-%m-%d")
    return t.to_csv(index=False)

_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.Pow: operator.pow, ast.USub: operator.neg}
def calculator(expression: str) -> str:
    # Safe arithmetic only: numbers and + - * / **
    def ev(n):
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.BinOp): return _OPS[type(n.op)](ev(n.left), ev(n.right))
        if isinstance(n, ast.UnaryOp): return _OPS[type(n.op)](ev(n.operand))
        raise ValueError("only arithmetic is allowed")
    return str(ev(ast.parse(expression, mode="eval").body))

TOOLS = {
    "search_documents": (search_documents, "search_documents(query): find passages in the Wikipedia articles about the laws"),
    "laws_table":       (laws_table,       "laws_table(): the structured table of laws (RA number, date signed, president) as CSV"),
    "calculator":       (calculator,       "calculator(expression): evaluate arithmetic like '2019 - 2012'"),
}

AGENT_PROMPT = '''You answer questions about Philippine laws by using tools, step by step.

TOOLS:
{tools}

At each step reply with ONLY one JSON object, nothing else. Either:
  {{"thought": "...", "tool": "<tool name>", "input": "<input>"}}
or, when you are done:
  {{"thought": "...", "final_answer": "<answer>"}}
Use the tools for facts and arithmetic; do not rely on memory.

QUESTION: {question}

{history}'''

def run_agent(question, tools=TOOLS, max_steps=6, verbose=True):
    history = ""
    tool_list = "\n".join(f"- {desc}" for _, desc in tools.values())
    for step in range(1, max_steps + 1):
        reply = llm(AGENT_PROMPT.format(tools=tool_list, question=question, history=history))
        try:
            action = parse_json(reply)
        except Exception:
            action = {}
        if "final_answer" in action:
            if verbose: print(f"✅ FINAL: {action['final_answer']}")
            return action["final_answer"]
        name, arg = action.get("tool"), action.get("input", "")
        if name in tools:
            try:
                observation = tools[name][0](arg)
            except Exception as e:
                observation = f"Tool error: {e}"
        else:
            observation = f"Unknown tool or bad format. Reply with JSON using one of: {list(tools)}"
        if verbose:
            print(f"Step {step} 🤔 {action.get('thought', '')}\n        🔧 {name}({arg!r})\n        👀 {str(observation)[:200]!r}…\n")
        history += f"STEP {step}: {json.dumps(action)}\nOBSERVATION: {str(observation)[:2000]}\n\n"
    return "Stopped: too many steps."

run_agent("How many years apart were the Data Privacy Act of 2012 and the Safe Spaces Act signed?");

In [ ]:
run_agent("Which law in our collection was signed most recently, and what does it do in one sentence?");

Read the trace. The model **planned**, **called functions**, and **combined** structured data (your Lab 2 table) with unstructured text (your Lab 3 vector DB). That's the shift from *models that predict* to *systems that act*.

> ⚠️ Agents multiply errors: a wrong row from Lab 2 becomes a wrong final answer here. Which is why Lab 4 matters.

---
## 🏁 Challenge: Group Work (30 minutes)
Pick **at least two** missions. At 3:30, share **your best score, one failure you found, and how you'd fix it.**

| Mission | Goal | Difficulty |
|---|---|---|
| **A · Beat the baseline** | Raise *RAG + ChromaDB* accuracy: change chunk size, overlap, `k`, `ADD_TITLE` or the prompt | ⭐ |
| **B · Break it** | Write 3 new test questions (a trap, a two-article question, a counting question) that make it fail | ⭐ |
| **C · New tool** | Give the agent a `years_since(date)` tool; ask *"How long has the SIM Registration Act been law?"* | ⭐⭐ |
| **D · Back to 2017** | Turn Notebook 1's dropout model into an agent tool, and have the agent explain one student's risk in plain words | ⭐⭐⭐ |

Tip: while tuning, evaluate only one system: `run_eval({"RAG + ChromaDB": SYSTEMS["RAG + ChromaDB"]})`.

### Mission A · Beat the baseline

In [ ]:
# 🧪 Change these, run the cell: it rebuilds chunks → embeddings → ChromaDB, then re-scores.
CHUNK_WORDS, OVERLAP, ADD_TITLE, K = 150, 30, True, 4

chunks = make_chunks(docs, CHUNK_WORDS, OVERLAP, ADD_TITLE)
E = embedder.encode(chunks.text.tolist(), normalize_embeddings=True)
collection = build_collection(chunks, E)
chroma_ret = ChromaRetriever(collection)
print(f"{len(chunks)} chunks · hit rate@{K}: {hit_rate(chroma_ret, K):.0%}")

mine = run_eval({"RAG + ChromaDB (mine)": lambda q: rag_answer(q, retriever=chroma_ret, k=K)["answer"]})
print(f"Accuracy: {mine.correct.mean():.0%}   (baseline: {scores.loc['RAG + ChromaDB', 'overall']:.0%})")

### Mission B · Break it

In [ ]:
MY_TESTS = pd.DataFrame([
    # ("question", ["acceptable", "answers"], "Source article title or None", "fact" or "trap"),
    ("How many of the laws in the collection were signed by Rodrigo Duterte?", ["5", "five"], None, "fact"),
    # add yours here…
], columns=["question", "accept", "source", "type"])

mine = run_eval({"RAG + ChromaDB": SYSTEMS["RAG + ChromaDB"]}, eval_df=MY_TESTS)
mine[["question", "answer", "correct"]]

In [ ]:
#@title 🔒 Solution ideas for Mission B { display-mode: "form" }
# Questions that tend to break simple RAG:
#  • Counting / aggregation: "How many of these laws did Duterte sign?" RAG retrieves passages; it can't count.
#      The agent with laws_table() can. Try: run_agent("How many of the laws were signed by Rodrigo Duterte?")
#  • Multi-hop: "Which president signed both the Safe Spaces Act and the Anti-Hazing Act?"
#      The facts live in two articles, and top-k may only retrieve one.
#  • Negation / traps: "Does the SIM Registration Act require registering email addresses?"
#  • Anything with "current": the snapshot is frozen in November 2026.
# Fixes: raise k, rewrite the query first, or route counting questions to the agent.

### Mission C · Give the agent a new tool

In [ ]:
# 🧪 Write the tool, register it, then ask the agent.
def years_since(date: str) -> str:
    ...  # your code: return how many years between `date` (YYYY-MM-DD) and today

# TOOLS["years_since"] = (years_since, "years_since(date): years from a YYYY-MM-DD date until today")
# run_agent("How long has the SIM Registration Act been law?")

In [ ]:
#@title 🔒 Solution for Mission C { display-mode: "form" }
from datetime import date as _date
def years_since(date: str) -> str:
    d = pd.to_datetime(date.strip()).date()
    return f"{(_date.today() - d).days / 365.25:.1f} years"

TOOLS["years_since"] = (years_since, "years_since(date): years from a YYYY-MM-DD date until today")
run_agent("How long has the SIM Registration Act been law?")

### Mission D · Back to 2017
The 2017 model predicts dropout risk. The 2026 version **explains** it to an adviser in plain language. Build it:
1. Load `student_dropout.csv` and train a quick random forest (copy ideas from Notebook 1).
2. Write a tool `student_risk(student_id)` that returns the predicted risk **and** the student's key numbers.
3. Register it and ask: *"Explain student 42's dropout risk to their adviser in two sentences, and suggest one action."*

In [ ]:
# 🧪 Your code here
students = pd.read_csv(io.BytesIO(fetch_data("student_dropout.csv")), sep=";")
students.columns = [c.strip() for c in students.columns]
students.head()

In [ ]:
#@title 🔒 Solution for Mission D { display-mode: "form" }
from sklearn.ensemble import RandomForestClassifier

S = students.copy()
S["dropout"] = (S["Target"] == "Dropout").astype(int)
S["pass_rate_1st"] = (S["Curricular units 1st sem (approved)"] / S["Curricular units 1st sem (enrolled)"].replace(0, np.nan)).fillna(0)
FEATURES = ["Age at enrollment", "Admission grade", "Tuition fees up to date", "Debtor", "Scholarship holder",
            "Curricular units 1st sem (approved)", "Curricular units 1st sem (grade)", "pass_rate_1st"]
rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, random_state=42).fit(S[FEATURES], S["dropout"])

def student_risk(student_id: str) -> str:
    row = S.loc[int(student_id)]
    p = rf.predict_proba(row[FEATURES].to_frame().T)[0, 1]
    facts = {f: (round(float(row[f]), 2)) for f in FEATURES}
    return json.dumps({"student_id": int(student_id), "dropout_risk": round(float(p), 2), "facts": facts})

TOOLS["student_risk"] = (student_risk, "student_risk(student_id): a student's predicted dropout risk (0-1) and key facts")
run_agent("Explain student 42's dropout risk to their adviser in two sentences, and suggest one action.")

---
## 🎓 What you just built
| You did… | Less time on… | More time on… |
|---|---|---|
| Loaded web articles as a dataset | tables only | **documents as data** |
| Extracted a table with an LLM, then *graded* it | hand-written parsers | **LLMs as data tools** |
| Embeddings, cosine similarity, ChromaDB, cited answers | standalone predictions | **interactive systems** |
| Measured retrieval, accuracy, refusals, the judge | model metrics only | **whole-system evaluation** |
| Gave an LLM tools and a loop | single answers | **systems that act** |

None of the right-hand column is new. Data scientists always cared about outcomes. AI just makes the left-hand work cheap enough that more of your time can go there.

Every step leaned on skills from Notebook 1: **pandas, vectors and similarity, test sets, and a skeptical eye.** AI didn't replace them. It made them more valuable.

**Keep going:** swap in documents you care about, try another embedding model, or rebuild this with LangChain or LlamaIndex now that you know what's underneath. Then put it on GitHub with an evaluation section.

*Data: Wikipedia (CC BY-SA 4.0), snapshot November 2026. Student dropout dataset: UCI Machine Learning Repository (CC BY 4.0), Realinho et al., 2021.*